# CNN 1D Multivariate

In this section we implement multivariate forecasting using the CNN 1D Model (1D Convolutional Neural Network) with the TimeSeriesDatasetVectorizedExog approach.

The CNN 1D (1D Convolutional Neural Network) Forecaster is the same univariate model used in the univariate approach, but extended to multivariate forecasting through efficient batching. Instead of processing series individually, TimeSeriesDatasetVectorizedExog batches all 1502 series together, allowing the univariate model to train on multiple series simultaneously with exogenous features (GDP, CPI, Interest Rate).

With this approach the model architecture remains unchanged - we simply reshape the data to process all series in parallel, achieving faster training while incorporating exogenous variables. Convolutional filters capture local temporal patterns across all series efficiently.

## Architecture

```bash
Input (seq_length, input_size)
    ↓
Transpose (input_size, seq_length)
    ↓
Conv1D Block 1
  ├─ Conv1D (input_size → hidden_size, kernel=3)
  ├─ ReLU
  └─ MaxPool1D
    ↓
Conv1D Block 2
  ├─ Conv1D (hidden_size → hidden_size, kernel=3)
  ├─ ReLU
  └─ MaxPool1D
    ↓
Conv1D Block 3
  └─ (same structure)
    ↓
Adaptive Average Pooling (→ 1)
    ↓
Flatten
    ↓
Fully Connected (hidden_size → hidden_size)
    ↓
ReLU + Dropout
    ↓
Fully Connected (hidden_size → 1)
    ↓
Output (1 prediction)
```

## Model

In [ ]:
import torch 
import torch.nn as nn

In [ ]:
class CNN1DForecaster(nn.Module):
    """
    1D CNN model for MULTIVARIATE time series forecasting.
    Architecture: 
        Conv1D blocks (Conv -> ReLU -> MaxPool) -> 
        Adaptive Average Pooling -> Flatten -> 
        Fully Connected -> Dropout -> Output
    
    CNNs can capture local patterns and temporal dependencies efficiently.
    Uses multiple kernel sizes to capture patterns at different scales.
    Processes sequences in parallel (unlike RNN/LSTM/GRU).
    """
    def __init__(self, input_size, hidden_size=64, num_layers=3, dropout=0.2):
        """
        Args:
            input_size: Number of input features (Value + year + month + one-hot)
            hidden_size: Number of filters in conv layers
            num_layers: Number of convolutional blocks (minimum 1)
            dropout: Dropout rate
        """
        super(CNN1DForecaster, self).__init__()
        
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = max(1, num_layers)
        
        # Convolutional layers
        conv_layers = []
        
        # First conv block: input_size -> hidden_size
        conv_layers.extend([
            nn.Conv1d(in_channels=input_size, out_channels=hidden_size, 
                     kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=1, padding=1)
        ])
        
        # Additional conv blocks: hidden_size -> hidden_size
        for i in range(1, self.num_layers):
            conv_layers.extend([
                nn.Conv1d(in_channels=hidden_size, out_channels=hidden_size, 
                         kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool1d(kernel_size=2, stride=1, padding=1)
            ])
        
        self.conv_blocks = nn.Sequential(*conv_layers)
        
        # Adaptive pooling to fixed size output
        self.adaptive_pool = nn.AdaptiveAvgPool1d(1)
        
        # Fully connected layers
        self.fc1 = nn.Linear(hidden_size, hidden_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        # x shape: (batch_size, seq_length, input_size)
        
        # Conv1d expects (batch_size, channels, seq_length)
        # Transpose from (batch, seq, features) to (batch, features, seq)
        x = x.transpose(1, 2)  # (batch_size, input_size, seq_length)
        
        # Apply convolutional blocks
        x = self.conv_blocks(x)  # (batch_size, hidden_size, seq_length')
        
        # Adaptive pooling to reduce to (batch_size, hidden_size, 1)
        x = self.adaptive_pool(x)  # (batch_size, hidden_size, 1)
        
        # Flatten
        x = x.squeeze(-1)  # (batch_size, hidden_size)
        
        # Fully connected layers
        x = self.fc1(x)  # (batch_size, hidden_size)
        x = self.relu(x)
        x = self.dropout(x)
        out = self.fc2(x)  # (batch_size, 1)
        
        return out


### Model Results without Exogenous Features

### Model Results with Exogenous Features